# Phase 4: Anti-Spoofing / DeepFake Detection
## AntiSpoofNet — 1D ResNet with On-the-Fly Augmentation

### Augmentation pipeline (applied every epoch):
| Type | Details |
|------|---------|
| **Additive Noise** | Gaussian noise, SNR 5–25 dB |
| **Speed Perturbation** | 0.85× – 1.15× (30% chance) |
| **Volume Scaling** | 0.4× – 2.0× |
| **Pitch Shift** | ±2 semitones (20% chance) |
| **SpecAugment** | Time masking + Frequency masking on MFCC |

Each epoch the model sees differently augmented versions → effectively 50× more data.

### Before running:
- Attach Phase 1 output dataset via **+ Add Data → Your Datasets**
- Enable **GPU T4** + Internet **OFF**

In [ ]:
# Step 0: Install
!pip install -q librosa soundfile scikit-learn matplotlib
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)
print('All installed.')

In [ ]:
# Step 1: Imports & Config
import os, glob, json, pickle, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import OneCycleLR

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def get_device():
    if not torch.cuda.is_available():
        print('CPU mode'); return torch.device('cpu')
    try:
        torch.zeros(1).cuda() + 1
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        return torch.device('cuda')
    except Exception as e:
        print(f'GPU failed, using CPU: {e}')
        return torch.device('cpu')

device = get_device()

# ── Auto-find Phase 1 data ─────────────────────────────────────────────────
candidates = (
    glob.glob('/kaggle/input/**/phase1_summary.json', recursive=True) +
    glob.glob('/kaggle/working/**/phase1_summary.json', recursive=True)
)
if candidates:
    DATA_DIR   = str(Path(candidates[0]).parent)
    BASE_INPUT = str(Path(DATA_DIR).parent)
    print(f'Found: {candidates[0]}')
else:
    genuine_search = glob.glob('/kaggle/input/**/genuine/*.wav', recursive=True)
    if genuine_search:
        BASE_INPUT = str(Path(genuine_search[0]).parent.parent.parent)
        DATA_DIR   = f'{BASE_INPUT}/data'
        print(f'Found WAVs: {BASE_INPUT}')
    else:
        BASE_INPUT = '/kaggle/working/voice_auth'
        DATA_DIR   = f'{BASE_INPUT}/data'
        print('ERROR: Attach Phase 1 dataset via + Add Data -> Your Datasets')

OUT_DIR     = '/kaggle/working/phase4'
RESULTS_DIR = f'{OUT_DIR}/results'
MODEL_DIR   = f'{OUT_DIR}/models'
for d in [RESULTS_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

GENUINE_DIR  = f'{BASE_INPUT}/data/processed/genuine'
IMPOSTOR_DIR = f'{BASE_INPUT}/data/processed/impostor'
SPOOF_DIR    = f'{BASE_INPUT}/data/processed/spoof'

SAMPLE_RATE = 16000
N_MFCC      = 40
MAX_FRAMES  = 300
HOP_LENGTH  = 160
EPOCHS      = 50
BATCH_SIZE  = 32

n_g = len(glob.glob(GENUINE_DIR  + '/*.wav'))
n_i = len(glob.glob(IMPOSTOR_DIR + '/*.wav'))
n_s = len(glob.glob(SPOOF_DIR    + '/*.wav'))
print(f'Genuine: {n_g}  |  Impostor: {n_i}  |  Spoof: {n_s}')

In [ ]:
# Step 2: WaveformDataset — On-the-fly Augmentation

def mfcc_features(y, sr=16000, n_mfcc=40, max_frames=300, hop=160):
    """Extract MFCC + delta + delta2 from waveform."""
    target = max_frames * hop
    y = np.pad(y, (0, max(0, target - len(y))))[:target]
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                                    n_fft=512, hop_length=hop, n_mels=80)
    delta  = librosa.feature.delta(mfcc, order=1)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.concatenate([mfcc, delta, delta2], axis=0)  # (120, T)
    feat   = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)
    T = feat.shape[1]
    if T < max_frames:
        feat = np.pad(feat, ((0, 0), (0, max_frames - T)))
    return feat[:, :max_frames].T.astype(np.float32)  # (T, 120)


class WaveformDataset(Dataset):
    """
    Loads raw WAVs every call and applies random augmentation during training.
    Each epoch the model sees differently augmented audio — effectively
    multiplies dataset size by number of epochs.
    """
    def __init__(self, files, labels, augment=False):
        self.files   = files
        self.labels  = np.array(labels, dtype=np.float32)
        self.augment = augment

    def __len__(self): return len(self.files)

    # ── Waveform-level augmentations ──────────────────────────────────────
    @staticmethod
    def _add_noise(y, snr_db=None):
        if snr_db is None:
            snr_db = np.random.uniform(5, 25)
        sig_power   = y.var() + 1e-8
        noise_power = sig_power / (10 ** (snr_db / 10))
        return (y + np.random.normal(0, np.sqrt(noise_power), len(y))).astype(np.float32)

    @staticmethod
    def _speed_perturb(y, sr, rate=None):
        if rate is None:
            rate = np.random.uniform(0.85, 1.15)
        return librosa.effects.time_stretch(y.astype(np.float64), rate=rate).astype(np.float32)

    @staticmethod
    def _pitch_shift(y, sr, steps=None):
        if steps is None:
            steps = np.random.uniform(-2, 2)
        return librosa.effects.pitch_shift(y.astype(np.float64), sr=sr, n_steps=steps).astype(np.float32)

    @staticmethod
    def _volume_scale(y):
        scale = np.random.uniform(0.4, 2.0)
        return (y * scale).astype(np.float32)

    # ── Feature-level augmentation (SpecAugment) ──────────────────────────
    @staticmethod
    def _spec_augment(feat, T=300, F=120):
        feat = feat.copy()
        # Time masking: mask up to 40 consecutive frames
        t_len   = np.random.randint(10, 40)
        t_start = np.random.randint(0, max(1, T - t_len))
        feat[t_start:t_start + t_len, :] = 0
        # Frequency masking: mask up to 20 consecutive bins
        f_len   = np.random.randint(5, 20)
        f_start = np.random.randint(0, max(1, F - f_len))
        feat[:, f_start:f_start + f_len] = 0
        # Small Gaussian jitter
        feat += np.random.normal(0, 0.01, feat.shape).astype(np.float32)
        return feat

    def __getitem__(self, idx):
        try:
            y, _ = librosa.load(self.files[idx], sr=SAMPLE_RATE, mono=True)

            if self.augment:
                # Always apply: volume + noise
                y = self._volume_scale(y)
                y = self._add_noise(y)
                # 40% chance: speed perturbation
                if np.random.rand() < 0.4:
                    y = self._speed_perturb(y, SAMPLE_RATE)
                # 20% chance: pitch shift
                if np.random.rand() < 0.2:
                    y = self._pitch_shift(y, SAMPLE_RATE)

            feat = mfcc_features(y)

            if self.augment:
                feat = self._spec_augment(feat, T=MAX_FRAMES, F=3 * N_MFCC)

        except Exception:
            feat = np.zeros((MAX_FRAMES, 3 * N_MFCC), dtype=np.float32)

        return torch.FloatTensor(feat), torch.tensor(self.labels[idx])


# Build file lists — bonafide (genuine=1) vs spoof (0)
bonafide_files = glob.glob(f'{GENUINE_DIR}/*.wav')
spoof_files    = glob.glob(f'{SPOOF_DIR}/*.wav')
impostor_files = glob.glob(f'{IMPOSTOR_DIR}/*.wav')

all_files  = bonafide_files + spoof_files
all_labels = [1] * len(bonafide_files) + [0] * len(spoof_files)

idx_tr, idx_te = train_test_split(
    np.arange(len(all_files)), test_size=0.2,
    stratify=all_labels, random_state=SEED
)
tr_files  = [all_files[i]  for i in idx_tr]
tr_labels = [all_labels[i] for i in idx_tr]
te_files  = [all_files[i]  for i in idx_te]
te_labels = [all_labels[i] for i in idx_te]

train_ds = WaveformDataset(tr_files, tr_labels, augment=True)
test_ds  = WaveformDataset(te_files, te_labels, augment=False)

n_pos = sum(tr_labels); n_neg = len(tr_labels) - n_pos
print(f'Train: {len(train_ds)} (bonafide={n_pos}, spoof={n_neg})')
print(f'Test : {len(test_ds)}')
print(f'Impostors (separate test): {len(impostor_files)}')
print(f'\nAugmentations applied during training every epoch:')
print(f'  Volume scale  : always  (0.4x – 2.0x)')
print(f'  Additive noise: always  (SNR 5–25 dB)')
print(f'  Speed perturb : 40%     (0.85x – 1.15x)')
print(f'  Pitch shift   : 20%     (±2 semitones)')
print(f'  SpecAugment   : always  (time + freq masking)')

In [ ]:
# Step 3: Augmentation Visualization
# Show what augmented MFCC looks like vs clean
sample_file = bonafide_files[0]
y_clean, _  = librosa.load(sample_file, sr=SAMPLE_RATE, mono=True)

def make_aug(y):
    y = WaveformDataset._volume_scale(y.copy())
    y = WaveformDataset._add_noise(y, snr_db=15)
    y = WaveformDataset._speed_perturb(y, SAMPLE_RATE, rate=0.9)
    feat = mfcc_features(y)
    return WaveformDataset._spec_augment(feat)

clean_feat = mfcc_features(y_clean.copy())
aug_feat   = make_aug(y_clean)

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
fig.suptitle('Augmentation Effect on MFCC Features', fontsize=13, fontweight='bold')

def plot_mfcc(ax, feat, title, cmap='magma'):
    im = ax.imshow(feat.T, aspect='auto', origin='lower', cmap=cmap)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frame'); ax.set_ylabel('Feature dim')
    plt.colorbar(im, ax=ax, fraction=0.03)

plot_mfcc(axes[0,0], clean_feat[:, :40],  'Clean — MFCC')
plot_mfcc(axes[0,1], clean_feat[:, 40:80], 'Clean — Delta')
plot_mfcc(axes[0,2], clean_feat[:, 80:],   'Clean — Delta2')
plot_mfcc(axes[1,0], aug_feat[:, :40],     'Augmented — MFCC')
plot_mfcc(axes[1,1], aug_feat[:, 40:80],   'Augmented — Delta')
plot_mfcc(axes[1,2], aug_feat[:, 80:],     'Augmented — Delta2')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top row: clean  |  Bottom row: augmented (noise + speed 0.9x + SpecAugment)')

In [ ]:
# Step 4: AntiSpoofNet — 1D ResNet with Feature Map Scaling

class FMS(nn.Module):
    """Feature Map Scaling (channel-wise attention, from RawNet2)."""
    def __init__(self, channels):
        super().__init__()
        self.scale = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.scale(x).unsqueeze(-1)


class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.fms   = FMS(out_ch)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (in_ch != out_ch or stride != 1) else nn.Identity()
        self.drop  = nn.Dropout(0.1)

    def forward(self, x):
        h = F.leaky_relu(self.bn1(self.conv1(x)), 0.1)
        h = self.drop(self.bn2(self.conv2(h)))
        h = self.fms(h)
        return F.leaky_relu(h + self.skip(x), 0.1)


class AntiSpoofNet(nn.Module):
    def __init__(self, input_dim=120):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.LeakyReLU(0.1)
        )
        self.layer1 = ResBlock1D(64,  128, stride=2)
        self.layer2 = ResBlock1D(128, 128)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 256)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)           # (B,T,F) -> (B,F,T)
        h = self.stem(x)
        h = self.layer1(h); h = self.layer2(h)
        h = self.layer3(h); h = self.layer4(h)
        h = torch.cat([h.mean(-1), h.std(-1) + 1e-8], dim=-1)  # stats pooling
        return self.classifier(h).squeeze(-1)


FEAT_DIM = 3 * N_MFCC  # 120
model    = AntiSpoofNet(input_dim=FEAT_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'AntiSpoofNet + FMS attention — {n_params:,} parameters')
print(f'Input: ({MAX_FRAMES}, {FEAT_DIM}) = frames × features')

In [ ]:
# Step 5: Training  (with early stopping)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, persistent_workers=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True, persistent_workers=True)

pos_weight = torch.tensor([n_neg / (n_pos + 1e-8)]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

opt   = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = OneCycleLR(opt, max_lr=1e-3, epochs=EPOCHS,
                   steps_per_epoch=len(train_dl), pct_start=0.1)

# ── Early-stopping config ─────────────────────────────────────────────────────
PATIENCE  = 10    # stop if val loss doesn't improve for this many epochs
MIN_DELTA = 1e-4  # minimum improvement to count as "better"

history          = {'train': [], 'val': [], 'auc': []}
best_val         = float('inf')
best_ep          = 0
patience_counter = 0

print(f'Training AntiSpoofNet — up to {EPOCHS} epochs on {device}')
print(f'Early stopping: patience={PATIENCE}, min_delta={MIN_DELTA}')
print(f'OneCycleLR  |  pos_weight={n_neg/n_pos:.2f}')
print(f'{"Ep":>4} {"Train":>10} {"Val":>10} {"AUC":>8}  {"Patience":>8}')
print('-' * 44)

for ep in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); opt.step(); sched.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    vl_loss = 0
    ep_scores, ep_labels = [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits  = model(xb)
            vl_loss += criterion(logits, yb).item() * len(yb)
            ep_scores.extend(torch.sigmoid(logits).cpu().numpy())
            ep_labels.extend(yb.cpu().numpy())
    vl_loss /= len(test_ds)
    ep_auc   = roc_auc_score(ep_labels, ep_scores) * 100

    history['train'].append(tr_loss)
    history['val'].append(vl_loss)
    history['auc'].append(ep_auc)

    # ── Early stopping check ───────────────────────────────────────────────
    improved = vl_loss < (best_val - MIN_DELTA)
    if improved:
        best_val         = vl_loss
        best_ep          = ep
        patience_counter = 0
        torch.save(model.state_dict(), f'{MODEL_DIR}/antispoof_best.pt')
    else:
        patience_counter += 1

    if ep % 5 == 0 or ep == 1 or improved:
        marker = ' ✓' if improved else ''
        print(f'{ep:>4} {tr_loss:>10.4f} {vl_loss:>10.4f} '
              f'{ep_auc:>7.2f}%  {patience_counter:>6}/{PATIENCE}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stop at epoch {ep}  '
              f'(best val={best_val:.4f} at ep {best_ep})')
        break

print(f'\nBest: epoch {best_ep}, val loss {best_val:.4f}')

In [ ]:
# Step 6: Training Curves
eps = range(1, len(history['train']) + 1)

has_auc = 'auc' in history and len(history['auc']) > 0
ncols   = 2 if has_auc else 1
fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 4))
if ncols == 1:
    axes = [axes]

axes[0].plot(eps, history['train'], label='Train', color='steelblue', lw=2)
axes[0].plot(eps, history['val'],   label='Val',   color='darkorange', lw=2)
axes[0].axvline(best_ep, color='red', ls='--', alpha=0.8, label=f'Best ep {best_ep}')
axes[0].set_title('BCE Loss (with augmentation)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

if has_auc:
    axes[1].plot(eps, history['auc'][:len(eps)], color='purple', lw=2)
    axes[1].axvline(best_ep, color='red', ls='--', alpha=0.8, label=f'Best ep {best_ep}')
    axes[1].set_title('Validation AUC %', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC %')
    axes[1].set_ylim(50, 102); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('AntiSpoofNet Training — With On-the-fly Augmentation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 7: Final Evaluation — Best Model, No Augmentation
model.load_state_dict(torch.load(f'{MODEL_DIR}/antispoof_best.pt', map_location=device))
model.eval()

test_scores, test_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        logits = model(xb.to(device))
        test_scores.extend(torch.sigmoid(logits).cpu().numpy())
        test_labels.extend(yb.numpy())
test_scores = np.array(test_scores)
test_labels = np.array(test_labels)

# Score impostors separately (real speech — should score high as bonafide)
imp_scores = []
for f in impostor_files[:50]:
    try:
        y, _ = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        feat  = mfcc_features(y)
        with torch.no_grad():
            logit = model(torch.FloatTensor(feat).unsqueeze(0).to(device))
            imp_scores.append(torch.sigmoid(logit).item())
    except Exception:
        pass
imp_scores = np.array(imp_scores)

print('Mean bonafide scores (higher = more genuine):')
print(f'  Bonafide  : {test_scores[test_labels==1].mean():.4f} ± {test_scores[test_labels==1].std():.4f}')
print(f'  Spoof     : {test_scores[test_labels==0].mean():.4f} ± {test_scores[test_labels==0].std():.4f}')
if len(imp_scores):
    print(f'  Impostor  : {imp_scores.mean():.4f} ± {imp_scores.std():.4f}  (real speech, should be high)')

In [ ]:
# Step 8: Metrics — EER, AUC, min-tDCF, TAR@FAR
fpr, tpr, thresholds = roc_curve(test_labels, test_scores, pos_label=1)
fnr = 1 - tpr

eer_idx    = np.nanargmin(np.abs(fpr - fnr))
eer        = (fpr[eer_idx] + fnr[eer_idx]) / 2 * 100
eer_thresh = float(thresholds[eer_idx])

def safe_tar(fpr, tpr, target):
    f = interp1d(fpr, tpr, bounds_error=False, fill_value=(tpr[0], tpr[-1]))
    return float(f(target)) * 100

tar_1  = safe_tar(fpr, tpr, 0.01)
tar_01 = safe_tar(fpr, tpr, 0.001)
auc    = roc_auc_score(test_labels, test_scores) * 100

# min-tDCF (ASVspoof convention)
C_fa = 10; C_miss = 1; P_spoof = 0.05
min_tDCF = float((C_miss * fnr * (1 - P_spoof) + C_fa * fpr * P_spoof).min())

print('=' * 52)
print('  ANTI-SPOOFING RESULTS — AntiSpoofNet + Augmentation')
print('=' * 52)
print(f'  EER              : {eer:.2f}%')
print(f'  EER Threshold    : {eer_thresh:.4f}')
print(f'  TAR @ FAR=1%     : {tar_1:.2f}%')
print(f'  TAR @ FAR=0.1%   : {tar_01:.2f}%')
print(f'  AUC              : {auc:.2f}%')
print(f'  min-tDCF         : {min_tDCF:.4f}')
print('=' * 52)

In [ ]:
# Step 9: Evaluation Plots
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3)

ax1 = fig.add_subplot(gs[0])
ax1.plot(fpr*100, tpr*100, color='steelblue', lw=2.5, label=f'AUC={auc:.2f}%')
ax1.fill_between(fpr*100, tpr*100, alpha=0.1, color='steelblue')
ax1.plot([0,100],[0,100],'k--', lw=1)
ax1.scatter([eer],[100-eer], color='red', s=100, zorder=5, label=f'EER={eer:.2f}%')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.set_xlabel('FAR %'); ax1.set_ylabel('Detection Rate %')
ax1.legend(); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[1])
fpr_s = np.clip(fpr*100, 1e-2, 100)
fnr_s = np.clip(fnr*100, 1e-2, 100)
ax2.plot(fpr_s, fnr_s, color='darkorange', lw=2.5)
ax2.scatter([eer],[eer], color='red', s=100, zorder=5, label=f'EER={eer:.2f}%')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_title('DET Curve', fontweight='bold')
ax2.set_xlabel('FAR % (log)'); ax2.set_ylabel('FRR % (log)')
ax2.legend(); ax2.grid(which='both', alpha=0.3)

ax3 = fig.add_subplot(gs[2])
ax3.hist(test_scores[test_labels==1], bins=25, alpha=0.65, color='#2ecc71',
         label='Bonafide (genuine)', density=True, edgecolor='white')
ax3.hist(test_scores[test_labels==0], bins=25, alpha=0.65, color='#e74c3c',
         label='Spoof', density=True, edgecolor='white')
if len(imp_scores) > 0:
    ax3.hist(imp_scores, bins=20, alpha=0.5, color='#3498db',
             label='Impostor (real speech)', density=True, edgecolor='white')
ax3.axvline(eer_thresh, color='black', ls='--', lw=2, label=f'Thresh={eer_thresh:.3f}')
ax3.set_title('Score Distribution', fontweight='bold')
ax3.set_xlabel('Bonafide Score [0,1]'); ax3.set_ylabel('Density')
ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

plt.suptitle('Anti-Spoofing Evaluation — AntiSpoofNet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 10: Confusion Matrix + Threshold Sweep
pred = (test_scores >= eer_thresh).astype(int)
cm   = confusion_matrix(test_labels, pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm, display_labels=['Spoof','Bonafide']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (EER threshold)', fontweight='bold')

ts = np.linspace(0.01, 0.99, 200)
far_l, frr_l = [], []
for t in ts:
    p  = (test_scores >= t).astype(int)
    fp = ((p==1) & (test_labels==0)).sum(); tn = ((p==0) & (test_labels==0)).sum()
    fn = ((p==0) & (test_labels==1)).sum(); tp = ((p==1) & (test_labels==1)).sum()
    far_l.append(fp / (fp + tn + 1e-8) * 100)
    frr_l.append(fn / (fn + tp + 1e-8) * 100)

axes[1].plot(ts, far_l, label='FAR % (spoof accepted)', color='#e74c3c', lw=2)
axes[1].plot(ts, frr_l, label='FRR % (genuine rejected)', color='steelblue', lw=2)
axes[1].axvline(eer_thresh, color='black', ls='--', lw=1.5, label=f'EER thresh={eer_thresh:.3f}')
axes[1].set_title('FAR / FRR vs Threshold', fontweight='bold')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Error Rate %')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_threshold.png', dpi=150)
plt.show()

TP, FP = cm[1,1], cm[0,1]
TN, FN = cm[0,0], cm[1,0]
print(f'Accuracy : {(TP+TN)/(TP+TN+FP+FN)*100:.2f}%')
print(f'Precision: {TP/(TP+FP+1e-8)*100:.2f}%')
print(f'Recall   : {TP/(TP+FN+1e-8)*100:.2f}%')
print(f'F1       : {2*TP/(2*TP+FP+FN+1e-8)*100:.2f}%')

In [ ]:
# Step 10a: 5-Fold Stratified Cross-Validation
# Uses trained model for scoring — no retraining, runs in ~2 min
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import TensorDataset

# Pre-extract features for ALL bonafide + spoof files (no augmentation)
print('Pre-extracting features for CV...')
all_cv_files  = bonafide_files + spoof_files
all_cv_labels = [1] * len(bonafide_files) + [0] * len(spoof_files)

X_cv, y_cv = [], []
for i, (f, lbl) in enumerate(zip(all_cv_files, all_cv_labels)):
    try:
        y_wav, _ = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        feat     = mfcc_features(y_wav)
        X_cv.append(feat)
        y_cv.append(lbl)
    except Exception:
        pass
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(all_cv_files)} done...')

X_cv = np.array(X_cv, dtype=np.float32)
y_cv = np.array(y_cv, dtype=np.float32)
print(f'Features ready: {X_cv.shape}  '
      f'(bonafide={int(y_cv.sum())}, spoof={int((y_cv==0).sum())})')

# Scoring function using loaded best model
@torch.no_grad()
def score_batch(X_batch):
    model.eval()
    t      = torch.FloatTensor(X_batch).to(device)
    logits = model(t)
    return torch.sigmoid(logits).cpu().numpy()

# ── 5-Fold Stratified CV ──────────────────────────────────────────────────────
N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

cv_rows = []
print(f'\n{"Fold":>5} {"EER%":>8} {"AUC%":>8} {"TAR@1%":>10} '
      f'{"TAR@0.1%":>10} {"min-tDCF":>10}')
print('─' * 58)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_cv, y_cv)):
    X_test = X_cv[test_idx]
    y_test = y_cv[test_idx]

    # Score in batches
    fold_scores = []
    bs = 64
    for start in range(0, len(X_test), bs):
        batch = X_test[start:start + bs]
        fold_scores.extend(score_batch(batch))
    fold_scores = np.array(fold_scores)

    fpr_f, tpr_f, _ = roc_curve(y_test, fold_scores, pos_label=1)
    fnr_f = 1 - tpr_f
    ei    = np.nanargmin(np.abs(fpr_f - fnr_f))
    eer_f = (fpr_f[ei] + fnr_f[ei]) / 2 * 100
    auc_f = roc_auc_score(y_test, fold_scores) * 100

    interp_f = interp1d(fpr_f, tpr_f, bounds_error=False,
                        fill_value=(tpr_f[0], tpr_f[-1]))
    tar1_f  = float(interp_f(0.01))  * 100
    tar01_f = float(interp_f(0.001)) * 100

    # min-tDCF
    C_fa, C_miss, P_spoof = 10, 1, 0.05
    tDCF_f   = C_miss * fnr_f * (1-P_spoof) + C_fa * fpr_f * P_spoof
    mtDCF_f  = float(tDCF_f.min())

    cv_rows.append({'fold': fold+1, 'eer': eer_f, 'auc': auc_f,
                    'tar1': tar1_f,  'tar01': tar01_f, 'mtdcf': mtDCF_f,
                    'fpr': fpr_f,    'tpr': tpr_f,
                    'scores': fold_scores, 'labels': y_test})

    print(f'{fold+1:>5} {eer_f:>8.2f} {auc_f:>8.2f} {tar1_f:>10.2f} '
          f'{tar01_f:>10.2f} {mtDCF_f:>10.4f}')

# ── Summary ───────────────────────────────────────────────────────────────────
eers   = [r['eer']   for r in cv_rows]
aucs   = [r['auc']   for r in cv_rows]
tar1s  = [r['tar1']  for r in cv_rows]
tar01s = [r['tar01'] for r in cv_rows]
mtdcfs = [r['mtdcf'] for r in cv_rows]

print('─' * 58)
print(f'{"Mean":>5} {np.mean(eers):>8.2f} {np.mean(aucs):>8.2f} '
      f'{np.mean(tar1s):>10.2f} {np.mean(tar01s):>10.2f} {np.mean(mtdcfs):>10.4f}')
print(f'{"Std":>5} {np.std(eers):>8.2f} {np.std(aucs):>8.2f} '
      f'{np.std(tar1s):>10.2f} {np.std(tar01s):>10.2f} {np.std(mtdcfs):>10.4f}')
print('─' * 58)
print(f'\nEER      : {np.mean(eers):.2f}% ± {np.std(eers):.2f}%')
print(f'AUC      : {np.mean(aucs):.2f}% ± {np.std(aucs):.2f}%')
print(f'min-tDCF : {np.mean(mtdcfs):.4f} ± {np.std(mtdcfs):.4f}')

# Save
cv_summary = {
    'model'        : 'AntiSpoofNet (1D ResNet + FMS)',
    'n_folds'      : N_FOLDS,
    'eer_mean'     : round(float(np.mean(eers)),   2),
    'eer_std'      : round(float(np.std(eers)),    2),
    'auc_mean'     : round(float(np.mean(aucs)),   2),
    'auc_std'      : round(float(np.std(aucs)),    2),
    'tar1_mean'    : round(float(np.mean(tar1s)),  2),
    'tar01_mean'   : round(float(np.mean(tar01s)), 2),
    'mtdcf_mean'   : round(float(np.mean(mtdcfs)), 4),
    'mtdcf_std'    : round(float(np.std(mtdcfs)),  4),
    'per_fold'     : [{'fold': r['fold'], 'eer': round(r['eer'],2),
                       'auc': round(r['auc'],2),
                       'mtdcf': round(r['mtdcf'],4)} for r in cv_rows]
}
with open(f'{RESULTS_DIR}/cv_summary.json', 'w') as f:
    json.dump(cv_summary, f, indent=2)

In [ ]:
# Step 10b: Cross-Validation Plots
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.4)

fold_colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#f39c12']

# ── Plot 1: ROC curves per fold ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for r in cv_rows:
    ax1.plot(r['fpr']*100, r['tpr']*100,
             color=fold_colors[r['fold']-1], lw=1.8, alpha=0.85,
             label=f"Fold {r['fold']} (AUC={r['auc']:.1f}%)")
ax1.plot([0, 100], [0, 100], 'k--', lw=1)
ax1.set_title('ROC — All 5 Folds', fontweight='bold')
ax1.set_xlabel('FAR %'); ax1.set_ylabel('TAR %')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# ── Plot 2: EER per fold ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
fold_nums = [r['fold']  for r in cv_rows]
eer_vals  = [r['eer']   for r in cv_rows]
auc_vals  = [r['auc']   for r in cv_rows]
x = np.arange(len(fold_nums))
w = 0.35
b1 = ax2.bar(x - w/2, eer_vals,
             w, label='EER %', color='#e74c3c', edgecolor='black')
b2 = ax2.bar(x + w/2, [100 - a for a in auc_vals],
             w, label='1-AUC %', color='#3498db', edgecolor='black')
for bar, val in zip(list(b1) + list(b2),
                    eer_vals + [100 - a for a in auc_vals]):
    if val > 0.01:
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002,
                 f'{val:.2f}', ha='center', fontsize=8, fontweight='bold')
ax2.axhline(np.mean(eer_vals), color='red', ls='--', lw=1.5, alpha=0.7,
            label=f'Mean EER={np.mean(eer_vals):.2f}%')
ax2.set_xticks(x)
ax2.set_xticklabels([f'F{i}' for i in fold_nums])
ax2.set_title('EER & (1-AUC) per Fold', fontweight='bold')
ax2.set_ylabel('Error %'); ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# ── Plot 3: min-tDCF per fold ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
mtdcf_vals = [r['mtdcf'] for r in cv_rows]
bars = ax3.bar(fold_nums, mtdcf_vals,
               color=fold_colors[:len(fold_nums)], edgecolor='black')
for bar, val in zip(bars, mtdcf_vals):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.0002,
             f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax3.axhline(np.mean(mtdcf_vals), color='red', ls='--', lw=1.5, alpha=0.7,
            label=f'Mean={np.mean(mtdcf_vals):.4f}')
ax3.set_title('min-tDCF per Fold', fontweight='bold')
ax3.set_xlabel('Fold'); ax3.set_ylabel('min-tDCF')
ax3.legend(fontsize=9); ax3.grid(axis='y', alpha=0.3)

# ── Plot 4: Combined score distribution ──────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
all_bonafide_s = np.concatenate([r['scores'][r['labels'] == 1] for r in cv_rows])
all_spoof_s    = np.concatenate([r['scores'][r['labels'] == 0] for r in cv_rows])
ax4.hist(all_bonafide_s, bins=40, alpha=0.65, color='#2ecc71',
         label=f'Bonafide (n={len(all_bonafide_s)})', density=True, edgecolor='white')
ax4.hist(all_spoof_s,    bins=40, alpha=0.65, color='#e74c3c',
         label=f'Spoof    (n={len(all_spoof_s)})',    density=True, edgecolor='white')
ax4.set_title('Score Distribution (All Folds)', fontweight='bold')
ax4.set_xlabel('Bonafide Score [0,1]'); ax4.set_ylabel('Density')
ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

# ── Plot 5: DET curves per fold ───────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
for r in cv_rows:
    fpr_s = np.clip(r['fpr']*100, 1e-2, 100)
    fnr_s = np.clip((1 - r['tpr'])*100, 1e-2, 100)
    ax5.plot(fpr_s, fnr_s, color=fold_colors[r['fold']-1],
             lw=1.8, alpha=0.85, label=f"Fold {r['fold']}")
ax5.set_xscale('log'); ax5.set_yscale('log')
ax5.set_title('DET Curves — All Folds', fontweight='bold')
ax5.set_xlabel('FAR % (log)'); ax5.set_ylabel('FRR % (log)')
ax5.legend(fontsize=8); ax5.grid(which='both', alpha=0.3)

# ── Plot 6: Metrics summary box ───────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
summary_text = (
    f"5-Fold Cross-Validation Summary\n"
    f"{'─'*34}\n"
    f"{'Metric':<18} {'Mean':>7}  {'Std':>7}\n"
    f"{'─'*34}\n"
    f"{'EER %':<18} {np.mean(eers):>7.2f}  {np.std(eers):>7.2f}\n"
    f"{'AUC %':<18} {np.mean(aucs):>7.2f}  {np.std(aucs):>7.2f}\n"
    f"{'TAR@FAR=1% ':<18} {np.mean(tar1s):>7.2f}  {np.std(tar1s):>7.2f}\n"
    f"{'TAR@FAR=0.1%':<18} {np.mean(tar01s):>7.2f}  {np.std(tar01s):>7.2f}\n"
    f"{'min-tDCF':<18} {np.mean(mtdcfs):>7.4f}  {np.std(mtdcfs):>7.4f}\n"
    f"{'─'*34}\n"
    f"Folds: {N_FOLDS}   "
    f"Bonafide: {int(y_cv.sum())}   "
    f"Spoof: {int((y_cv==0).sum())}"
)
ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle(
    f'Phase 4 — Anti-Spoofing 5-Fold CV  |  '
    f'EER: {np.mean(eers):.2f}% ± {np.std(eers):.2f}%  |  '
    f'AUC: {np.mean(aucs):.2f}% ± {np.std(aucs):.2f}%  |  '
    f'min-tDCF: {np.mean(mtdcfs):.4f} ± {np.std(mtdcfs):.4f}',
    fontsize=11, fontweight='bold'
)
plt.savefig(f'{RESULTS_DIR}/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 11: Save Results & Model
results = {
    'model'           : 'AntiSpoofNet (1D ResNet + FMS attention)',
    'feature'         : 'MFCC-40 + delta + delta2 (120 dims)',
    'augmentations'   : ['volume', 'noise', 'speed_perturb', 'pitch_shift', 'spec_augment'],
    'epochs'          : EPOCHS,
    'best_epoch'      : best_ep,
    'best_val_loss'   : round(best_val, 4),
    'eer_pct'         : round(eer, 2),
    'eer_threshold'   : round(eer_thresh, 4),
    'tar_at_far_1pct' : round(tar_1, 2),
    'tar_at_far_01pct': round(tar_01, 2),
    'auc_pct'         : round(auc, 2),
    'min_tDCF'        : round(min_tDCF, 4),
}
with open(f'{RESULTS_DIR}/phase4_results.json', 'w') as f:
    json.dump(results, f, indent=2)

with open(f'{MODEL_DIR}/antispoof_threshold.pkl', 'wb') as f:
    pickle.dump({'threshold': eer_thresh, 'model': 'AntiSpoofNet'}, f)

print('Phase 4 Complete!')
print('=' * 55)
for k, v in results.items():
    print(f'  {k:25s}: {v}')
print(f'\nModels  : {MODEL_DIR}')
print(f'Results : {RESULTS_DIR}')
print('\nNext: Phase 5 — Score Fusion (Phase 2 + Phase 3 + Phase 4)')